# 08. Tracer initialization and its impact on the hindcasts

HYB_BASE started from the bare ERA5 `eam.i` (T, U, V, Q, PS, CLDLIQ, CLDICE): ozone, the linoz gases the radiation reads (`rad_climate`: H2OLNZ, O3, N2OLNZ, CH4LNZ), aerosols and chemistry start near zero. ERA5CLIMFILL is the same ERA5 state with the same DART perturbations and DATM-forced land, the missing fields taken from CLIM_BASE. **HYB_BASE - ERA5CLIMFILL is the effect of the tracer initialization.** It is followed along the chain tracers (ozone columns) → clear-sky radiation → stratospheric temperature → tropospheric circulation and moisture → surface temperature, and set against the initialization signals it would contaminate: DART - ERA5 atmosphere (`DA_ATM - ERA5CLIMFILL`) and ERA5 - free-running state (`ERA5CLIMFILL - CLIM_BASE`).

## Statistics

Members are exchangeable and unpaired across configurations; the starts are independent. For a contrast a - b: difference of the ensemble means averaged over the starts; standard error from the member variances, sqrt(Σ_starts (var_a/n_a + var_b/n_b)) / n_starts; member spread sqrt(mean of the two member variances); z = difference / standard error. A difference has **recovered** from the day after it last exceeds `recovery_k` × member spread; it is **significant** while |z| > `z_crit`. Skill (variables with a reference: ERA5, GPCP): ensemble-mean bias and RMSE, spread and spread/RMSE by region and lead day.

Settings: `TracerInitSettings` and `TRACER_CHAIN` in `configs/hind_lead_eval_cases.py`; products: `util/tracer_init_workflow.py`; methods and figures: `util/tracer_init.py`. Design of the hindcasts: `METHODS.md`.

Every product is saved with the settings it was computed from and read back on the next run when those settings are unchanged; `FORCE_COMPUTE = True` recomputes this notebook's results, `FORCE_INPUTS = True` also re-reads the model output and observations. Model output is read only when a product has to be (re)computed (the first run reads every member of every configuration, season and variable: slow; `CASE_OVERRIDES = {"max_members": 3}` for a quick test).

## Outputs

`data/hind_lead_eval/tracer_init/`: `summary_<CONFIG>_<SEASON>_<VAR>.nc`, `reference_<SEASON>_<VAR>.nc` (member reductions), `tracer_init_contrasts.csv`, `tracer_init_recovery.csv`, `tracer_init_skill.csv`, `tracer_init_zonal_<SEASON>.nc`, `tracer_init_maps_<SEASON>.nc`. Figures in `figure/hind_lead_eval/tracer_init/`: `series_<step>_<REGION>.png`, `contrasts_<step>_<REGION>.png`, `recovery_chain.png`, `zonal_<step>.png`, `maps_<VAR>.png`, `skill_<METRIC>_<REGION>.png`.

## 1. Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

## 2. User setup

Paths and run options; plain values only, so `papermill -p` overrides work. Case definitions (experiments, contrasts, variables, regions, windows, recovery thresholds) are the settings defaults; change them for this notebook with `CASE_OVERRIDES`.

In [ ]:
INPUT_DATA_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/eam_hindcast")
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/e3sm_dart/diag_dart_2026")
OBSERVATION_ROOT = Path("/global/cfs/cdirs/e3sm/zhan391/data")  # reference-data archive
LAND_SUPPORT_DIR = OBSERVATION_ROOT / "lnd_sea_mask"           # land fraction
AMIP_ROOT = OBSERVATION_ROOT / "E3SMv3"                        # free-running E3SMv3 DECK AMIP runs
WORKFLOW_NAME = "hind_lead_eval"

# Initialization configurations (configs/fcst_init_experiment_config.py, EXPERIMENTS). This notebook answers
# question (a): CONFIG_GROUP "tracer_init" = CLIM_BASE, HYB_BASE, ERA5CLIMFILL, DA_ATM. Another group or list must
# still contain the configurations of TracerInitSettings.test/control/contrasts (or change them in CASE_OVERRIDES).
# CASE_NAMES_OVERRIDE maps a selected key to another raw case name (INPUT_DATA_ROOT/<name>_<YYYYMMDD>00/<ENxx>).
CONFIG_GROUP = "tracer_init"
CASE_NAMES_OVERRIDE = {}

CASE_OVERRIDES = {}      # changes to the TracerInitSettings defaults, e.g. {"max_members": 3, "seasons": ("DJF",)}
FORCE_COMPUTE = False    # recompute this notebook's results even when they are saved with the same settings
FORCE_INPUTS = False     # also re-read the model output and observations (slow)

## 3. Paths and modules

In [ ]:
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.fcst_init_experiment_config import select_configs, set_case_names
from configs.hind_lead_eval_cases import TRACER_CHAIN, RunPaths, TracerInitSettings
from util.hind_case_io import figure_saver, write_table
from util.tracer_init_workflow import TracerInitCase

paths = RunPaths(*(Path(p) for p in (INPUT_DATA_ROOT, WORK_DIR, DIAGNOSTIC_OUTPUT_ROOT, OBSERVATION_ROOT,
                                     LAND_SUPPORT_DIR, AMIP_ROOT)), workflow_name=WORKFLOW_NAME)
CONFIGS = select_configs(CONFIG_GROUP, CASE_NAMES_OVERRIDE)   # {configuration key: raw case name}
set_case_names(CONFIGS)   # the readers find each configuration's cases by these names
for c, name in CONFIGS.items():
    print(f"[CASE] {c}: {name}_<YYYYMMDD>00")
case = TracerInitCase(TracerInitSettings(paths=paths, **{"configs": tuple(CONFIGS), **CASE_OVERRIDES}),
                      force_compute=FORCE_COMPUTE, force_inputs=FORCE_INPUTS)
cfg = case.cfg
display(case.experiments())

## 4. Read input

Checks that every configuration is post-processed for each season (`regen_ts_<stream>.status` OK for all members), then reduces every member of every configuration, season and variable of `TRACER_CHAIN` as it is read: regional and zonal means by lead day, lead-window mean maps and, where a reference exists, the regional error (`summary_*.nc`); and the regional and zonal means of the references (`reference_*.nc`). Cached files are read back.

In [ ]:
seasons = case.seasons
for step, variables in TRACER_CHAIN.items():
    print(f"[CHAIN] {step}: " + ", ".join(f"{v} [{case.units[v]}]{'' if case.verifiable(v) else ' (no reference)'}"
                                         for v in variables))
for s in seasons:
    for v in cfg.variables:
        for c in cfg.configs:
            case.summary(c, s, v)
        case.reference(s, v)

## 5. Process and save

* `contrasts`: difference, standard error, member spread and z of every contrast by region and lead day;
* `recovery`: day-1 and peak difference, the spread at the peak and the last day outside k × spread / significant (`max_lead`: not recovered within the run);
* `skill`: bias, RMSE, spread and spread/RMSE against the reference;
* `zonal`, `maps`: zonal-mean (latitude × lead) and lead-window map contrasts.

In [ ]:
contrasts = case.contrasts()
recovery = case.recovery()
skill = case.skill()
zonal = {s: case.zonal(s) for s in seasons}
MAP_VARIABLES = ("TOZ", "FLDSC", "T050", "Z500", "TREFHT")   # one per chain step
maps = {s: case.maps(s, MAP_VARIABLES) for s in seasons}

# the tracer defect along the chain: day-1 and peak difference, and the last day outside k x spread / significant
rows = [(v, r, s) for v, r in cfg.chain for s in seasons]
rec = recovery.set_index(["variable", "region", "season"])
display(rec[rec.contrast == "tracer_defect"].reindex(rows).round(3))

# the defect next to the initialization signals it would contaminate (peak difference, last day outside the spread)
side = recovery.pivot_table(index=["variable", "region", "season"], columns="contrast",
                            values=["peak", "outside_spread_until"], aggfunc="first")
display(side.reindex(rows).reindex(columns=list(cfg.contrasts), level="contrast").round(3))

## 6. Plot and show

In [ ]:
# ============================ Figure settings: edit here and rerun this cell ============================
SHOW = True          # False: save the figures without showing them
FIGURE_DPI = 200
FONT_SIZE = 18       # base font size (pt); every label of every figure scales with it
FONT_SCALES = dict(label=1.0, title=1.0, tick=0.9, legend=1.0, suptitle=1.15, cell=0.75)  # x FONT_SIZE
# region shown for each step of TRACER_CHAIN (series and contrast curves); zonal and map figures cover the globe
STEP_REGIONS = {"tracers": "GLOBAL", "radiation": "GLOBAL", "stratosphere": "GLOBAL",
                "troposphere": "NH_EXTRATROPICS", "surface": "LAND"}
SKILL_REGIONS = ("LAND", "NH_EXTRATROPICS")

from util.fcst_init_plotting import set_font_size
set_font_size(FONT_SIZE, **FONT_SCALES)
save = figure_saver(case.figure_dir, dpi=FIGURE_DPI, show=SHOW)
# =========================================================================================================

### 6.1 Regional means by lead day

Every configuration (mean over starts and members, 10-90 % of member-starts shaded) and the reference, one figure per chain step.

In [ ]:
from util.tracer_init import plot_regional_series

for step, variables in TRACER_CHAIN.items():
    r = STEP_REGIONS[step]
    series, obs = case.series(variables, r)
    save(plot_regional_series(series, obs, list(variables), seasons=seasons, region_label=case.region_labels[r],
                              units=case.units, max_lead=cfg.max_lead), f"series_{step}_{r}.png")

### 6.2 Contrasts by lead day

The tracer defect (black, ±`z_crit` standard errors; grey: ±k × member spread; dotted: recovery day) next to the initialization signals it would contaminate.

In [ ]:
from util.tracer_init import plot_contrast_curves

for step, variables in TRACER_CHAIN.items():
    r = STEP_REGIONS[step]
    save(plot_contrast_curves(contrasts, list(variables), contrasts=cfg.contrasts, seasons=seasons, region=r,
                              region_label=case.region_labels[r], units=case.units, k=cfg.recovery_k,
                              z_crit=cfg.z_crit, max_lead=cfg.max_lead), f"contrasts_{step}_{r}.png")

### 6.3 From tracers to the surface

The tracer defect of each (variable, region) of `cfg.chain`, normalized by its peak, with its recovery day: the order in which the steps of the chain respond and recover.

In [ ]:
from util.tracer_init import plot_recovery_chain

save(plot_recovery_chain(contrasts, cfg.chain, contrast="tracer_defect", pair=cfg.contrasts["tracer_defect"],
                         seasons=seasons, k=cfg.recovery_k, max_lead=cfg.max_lead), "recovery_chain.png")

### 6.4 Zonal-mean defect, latitude × lead day

`HYB_BASE - ERA5CLIMFILL`; contour: |Δ| = k × member spread.

In [ ]:
from util.tracer_init import contrast_label, plot_zonal_lead

pair = cfg.contrasts["tracer_defect"]
for step, variables in TRACER_CHAIN.items():
    panels = {(s, v): zonal[s].sel(contrast="tracer_defect", variable=v) for s in seasons for v in variables}
    save(plot_zonal_lead(panels, list(variables), seasons=seasons, units=case.units, k=cfg.recovery_k,
                         max_lead=cfg.max_lead, title=f"Zonal-mean {contrast_label(pair)}: {step}"),
         f"zonal_{step}.png")

### 6.5 Lead-window maps

Every contrast (columns) by lead window (rows), one figure per variable on one colour scale; stippled where |z| > `z_crit`.

In [ ]:
from util.fcst_init_plotting import plot_window_maps
from util.tracer_init import symmetric_levels

names = {n: contrast_label(p) for n, p in cfg.contrasts.items()}   # column titles
for v in MAP_VARIABLES:
    panels = {s: {names[n]: maps[s].delta.sel(contrast=n, variable=v) for n in cfg.contrasts} for s in seasons}
    stipple = {s: {names[n]: np.abs(maps[s].z.sel(contrast=n, variable=v)) > cfg.z_crit for n in cfg.contrasts}
               for s in seasons}
    levels = symmetric_levels(np.concatenate([maps[s].delta.sel(variable=v).values.ravel() for s in seasons]))
    save(plot_window_maps(panels, stipple=stipple, levels=levels, label=f"Δ{v} [{case.units[v]}]",
                          title=f"Δ{v}, lead-window means (stippled: |z| > {cfg.z_crit:g})"), f"maps_{v}.png")

### 6.6 Impact on skill

Ensemble-mean bias, RMSE, spread and spread/RMSE against the reference by lead day, and the % change of the lead-window mean RMSE of every configuration against the control (`ERA5CLIMFILL`): how much of HYB_BASE's error is the tracer defect.

In [ ]:
from util.fcst_init_plotting import plot_lead_metric
from util.tracer_init import window_change

datasets = case.skill_datasets(skill)
verifiable = [v for v in cfg.variables if case.verifiable(v)]
for r in SKILL_REGIONS:
    for metric in ("bias", "rmse", "spread_ratio"):
        save(plot_lead_metric(datasets, verifiable, metric=metric, seasons=seasons, region=r,
                              region_label=case.region_labels[r], max_lead=cfg.max_lead), f"skill_{metric}_{r}.png")
    for s, t in window_change(skill, cfg.control, cfg.lead_windows, region=r, variables=verifiable).items():
        print(f"Lead-window RMSE change vs {cfg.control} (%), {s}, {case.region_labels[r]}")
        display(t.round(1))